In [3]:
require(data.table)
require(ggplot2)
require(ggpubr)

subject_color_map = c(
  "FH1001" = "#1f77b4", "FH1002" = "#ff7f0e", "FH1003" = "#279e68",
  "FH1004" = "#d62728", "FH1005" = "#aa40fc", "FH1006" = "#8c564b",
  "FH1007" = "#e377c2", "FH1008" = "#b5bd61", "FH1009" = "#17becf",
  "FH1010" = "#aec7e8", "FH1011" = "#ffbb78", "FH1012" = "#98df8a",
  "FH1014" = "#ff9896", "FH1016" = "#c5b0d5", "FH1017" = "#c49c94",
  "FH1018" = "#f7b6d2", "FH1021" = "#dbdb8d"
)

Loading required package: data.table

Loading required package: ggplot2

Loading required package: ggpubr



In [4]:
### Load in COVID Data 
covidVaccine = fread('../data/msd/COVID_MSD.csv')

### re-label all 90 day visits to be 60 (only 1-2 donors)
### and then keep the first visit to standardize across different time points 
covidVaccine[`Visit Type` == 'MM Post Transplant 90 Days']$`Visit Type` <- 'MM Post Transplant 60 Days'
covidVaccine <- covidVaccine[order(Subject, `Visit Type`, DaysSinceFirstVisit), .SD[1], by = .(Subject, `Visit Type`)]

### remove excess columns
vacc_cols = grep('Mean|Subject|Visit', colnames(covidVaccine))

### filter to participants, visits and read outs
covidVaccine <- covidVaccine[, vacc_cols, with=FALSE]

In [5]:
## Define and subset to clinical time points
clinicalTimePoints = c('MM End Induction 1st Draw', 'MM Post Transplant 60 Days',
                       'MM Post Transplant 1 year', 'MM Post Transplant 2 year')

covidVaccine = covidVaccine[covidVaccine$`Visit Type` %in% clinicalTimePoints]
covidVaccine$Visit = covidVaccine$`Visit Type`

### We then clean up the naming conventions
### to align them closer to the rest of the manuscript
covidVaccine$Visit <- gsub('MM|1st Draw|','', covidVaccine$Visit)
covidVaccine$Visit <- gsub('Post Transplant','ASCT', covidVaccine$Visit)
covidVaccine$Visit <- gsub(' ','', covidVaccine$Visit)
covidVaccine$Visit <- gsub('ays|ear|','', covidVaccine$Visit)
covidVaccine$Visit <- gsub('EndInduction','End-Ind', covidVaccine$Visit)
covidVaccine <- covidVaccine[, !colnames(covidVaccine) %in% c('Visit Type','DaysSinceFirstVisit'), with=F]

### Remove empty NA row and melt from wide to long 
tmp = melt(covidVaccine, id.vars = c('Subject','Visit'))
tmp$value = as.numeric(tmp$value)
### Filter only to VRD Subjects 
tmp = tmp[Subject %in% paste('FH', c(1001:1021),sep='')]

### Factor visits into clinical order 
tmp$Visit <- factor(tmp$Visit, levels = c('End-Ind','ASCT60D','ASCT1y','ASCT2y'))

write.csv(tmp, '../data/msd/output/processed_covid_data.csv')